# 03 · Corridas: baseline completo y variantes

**Etapa 5 del plan.** Entrena **todas las configuraciones del pre-registro** (`docs/diseno_experimental.md`) con las mismas 10 semillas, elige la cantidad de envíos (N) en 201908 y guarda las predicciones de 201908, 201909 y 201910.

**Ceguera:** este notebook **no carga la clase de 201909 ni de 201910** (se lee en NULL desde el SQL). No calcula ganancia ni AUC de esos meses. Eso lo hace el `04`, una sola vez, aplicando la regla pre-registrada.

**Antes de correrlo:** el pre-registro y este notebook tienen que estar commiteados.

Las variables, Optuna y los modelos salen del `02`, ya verificado. Ficha: `fichas/etapa_5_corridas.md` (en el laboratorio).

## 1. Parámetros

- `CONFIGURACIONES`: cuáles se corren. **Reproducción cruzada** en la VM de un compañero: `['A']` y `N_SEMILLAS = 2`. Usa los hiperparámetros de `resultados/03_hiperparametros.json`, sin volver a correr Optuna.
- **Retomar:** si la corrida se corta, se vuelve a correr entera. Las pruebas de Optuna y las semillas ya hechas no se repiten.
- **Corrida de prueba:** `N_PRUEBAS_OPTUNA = 2`, `N_SEMILLAS = 1` y `ARBOLES_MAX = 200`, en un directorio aparte (`EXP` y `RESULTADOS`), para no mezclarla con la de verdad.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import resource
import sys
import time
import warnings

import duckdb
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import roc_auc_score

# --- Parámetros -------------------------------------------------------------
BASE_PARQUET = Path('../data/base_2019.parquet')
RESULTADOS = Path('../resultados')
EXP = Path('../exp/03')

CONFIGURACIONES = ['A', 'L3', 'L4', 'T3', 'T4', 'LT3', 'LT4', 'A-ranking', 'LT4-ranking', 'A-consigna']
MESES_TRAIN = [201905, 201906]                       # provisorio
MESES_TRAIN_CONSIGNA = [201903, 201904, 201905, 201906]
MES_VALID = 201908
MESES_TEST = [201909, 201910]                       # su clase NO se carga
TRATAR_ROTOS = True

SEMILLA_PRIMIGENIA = 979201
N_SEMILLAS = 10
POSITIVOS = ['BAJA+1', 'BAJA+2']                    # clase de entrenamiento (provisorio)
POSITIVOS_AUC = ['BAJA+2']

N_PRUEBAS_OPTUNA = 50
PRIMERA_PRUEBA_02 = True                            # decisión 2: la mejor del 02 como prueba 0
MEJOR_02 = {'num_leaves': 77, 'min_data_in_leaf': 1147, 'learning_rate': 0.01559691860629471,
            'feature_fraction': 0.5587233179353345, 'bagging_fraction': 0.9804939364542807}
ARBOLES_MAX = 3000
PACIENCIA = 100
N_HILOS = 8

GANANCIA_ACIERTO = 1_072_500                        # provisorio
COSTO_ESTIMULO = 27_500
VENTANA_SUAVIZADO = 501
LARGO_CURVA = 25_000
# -----------------------------------------------------------------------------

VARIANTES = {   # lags además de 1 y 2, y largo de la tendencia (ventana de m − h a m)
    'A':   {'lags': [],     'tendencia': None},
    'L3':  {'lags': [3],    'tendencia': None},
    'L4':  {'lags': [3, 4], 'tendencia': None},
    'T3':  {'lags': [],     'tendencia': 3},
    'T4':  {'lags': [],     'tendencia': 4},
    'LT3': {'lags': [3],    'tendencia': 3},
    'LT4': {'lags': [3, 4], 'tendencia': 4},
}
CONFIG = {   # configuración: (variante, meses de entrenamiento, escala de la historia)
    **{v: (v, MESES_TRAIN, 'nominal') for v in VARIANTES},
    'A-ranking': ('A', MESES_TRAIN, 'ranking'),
    'LT4-ranking': ('LT4', MESES_TRAIN, 'ranking'),
    'A-consigna': ('A', MESES_TRAIN_CONSIGNA, 'nominal'),
}
GRUPO_CONTROL = [c for c, (_, meses, hist) in CONFIG.items() if meses == MESES_TRAIN and hist == 'nominal']

PRED = EXP / 'pred'
CURVAS = EXP / 'curvas_201908'
STORAGE = f"sqlite:///{(EXP / 'optuna.db').as_posix()}"
CLAVES_EVAL = EXP / 'claves_eval.npz'
HASHES = EXP / 'hashes_control.json'
ROTOS_CSV = RESULTADOS / '01_meses_rotos.csv'
COBERTURA_CSV = RESULTADOS / '01_cobertura_historia.csv'
HIPER_JSON = RESULTADOS / '03_hiperparametros.json'
VALIDACION_CSV = RESULTADOS / '03_validacion.csv'
N_CSV = RESULTADOS / '03_N.csv'
TIEMPOS_CSV = RESULTADOS / '03_tiempos.csv'

warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14
optuna.logging.set_verbosity(optuna.logging.WARNING)
for d in (PRED, CURVAS, RESULTADOS):
    d.mkdir(parents=True, exist_ok=True)

assert BASE_PARQUET.exists(), f'Falta {BASE_PARQUET.resolve()}: correr antes el 01'
assert set(CONFIGURACIONES) <= set(CONFIG), set(CONFIGURACIONES) - set(CONFIG)
assert 1 <= N_SEMILLAS <= 10
assert max(MESES_TRAIN_CONSIGNA + MESES_TRAIN) < MES_VALID < min(MESES_TEST)
MESES_EVAL = [MES_VALID] + MESES_TEST


def p0(mes):
    """Período consecutivo: el mes anterior es siempre p0 − 1."""
    return (mes // 100) * 12 + mes % 100


TIEMPOS = []


def memoria_gb():
    """Memoria máxima del proceso hasta ahora (macOS la da en bytes; Linux, en KB)."""
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return round(r / (1e9 if sys.platform == 'darwin' else 1e6), 2)


def registrar(config, paso, t0):
    segundos = time.time() - t0
    TIEMPOS.append({'config': config, 'paso': paso, 'segundos': round(segundos, 1), 'memoria_max_gb': memoria_gb()})
    print(f'{config} · {paso}: {segundos:.0f} s · memoria máxima {memoria_gb()} GB')


print('Configuraciones:', CONFIGURACIONES)

Configuraciones: ['A', 'L3', 'L4', 'T3', 'T4', 'LT3', 'LT4', 'A-ranking', 'LT4-ranking', 'A-consigna']


## 2. Valores esperados y semillas

In [2]:
ESPERADO_FILAS_POR_MES = {
    201901: 123_916, 201902: 125_020, 201903: 125_622, 201904: 126_116, 201905: 126_763,
    201906: 128_370, 201907: 129_853, 201908: 131_775, 201909: 133_419, 201910: 135_633,
}
ESPERADO_CLASES = {   # BAJA+1, BAJA+2 (solo hasta la validación)
    201901: (647, 697), 201902: (700, 696), 201903: (700, 774), 201904: (776, 575),
    201905: (576, 658), 201906: (665, 589), 201907: (595, 656), 201908: (659, 543),
}
ESPERADO_N_VARIABLES = {'A': 760, 'L3': 1_044, 'L4': 1_328, 'T3': 1_284, 'T4': 1_284, 'LT3': 1_568, 'LT4': 1_852}
ESPERADO_ROTOS = 23
ESPERADO_ROTOS_201910 = 12
VARIABLE_COBERTURA = 'cproductos'   # conteo sin nulos ni meses rotos


def primos_6_digitos():
    criba = np.ones(1_000_000, dtype=bool)
    criba[:2] = False
    for i in range(2, 1_001):
        if criba[i]:
            criba[i * i::i] = False
    return np.flatnonzero(criba[100_000:]) + 100_000


SEMILLAS = np.random.default_rng(SEMILLA_PRIMIGENIA).choice(primos_6_digitos(), size=20, replace=False).tolist()
SEMILLAS_A_USAR = SEMILLAS[:N_SEMILLAS]
assert SEMILLAS[:10] == [294103, 342929, 161093, 677561, 431077, 330679, 511579, 312773, 118147, 567529]
print('Semillas:', SEMILLAS_A_USAR)

Semillas: [294103, 342929, 161093, 677561, 431077, 330679, 511579, 312773, 118147, 567529]


## 3. Carga, con la clase de los meses de test en NULL

Se cargan los meses de 201901 a 201910. **La clase de 201909 y 201910 se reemplaza por NULL en el mismo SQL**, así que nunca llega a la memoria.

In [3]:
t0 = time.time()
lista_test = ', '.join(str(m) for m in MESES_TEST)
con = duckdb.connect()
base = con.execute(f"""
    select * exclude (clase_ternaria),
           case when foto_mes in ({lista_test}) then null else clase_ternaria end as clase_ternaria
    from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes <= {max(MESES_TEST)}
    order by foto_mes, numero_de_cliente
""").df()
con.close()

por_mes = base['foto_mes'].value_counts().sort_index().to_dict()
assert por_mes == ESPERADO_FILAS_POR_MES, por_mes
clases = base.groupby('foto_mes')['clase_ternaria'].value_counts().unstack()
for mes, (b1, b2) in ESPERADO_CLASES.items():
    assert (clases.loc[mes, 'BAJA+1'], clases.loc[mes, 'BAJA+2']) == (b1, b2), mes
es_test = base['foto_mes'].isin(MESES_TEST)
assert base.loc[es_test, 'clase_ternaria'].isna().all(), 'CEGUERA: la clase de los meses de test no puede estar cargada'
assert base.loc[~es_test, 'clase_ternaria'].notna().all()
registrar('-', 'carga', t0)
print(f'{len(base):,} filas · clase de {MESES_TEST}: 100 % NULL')

- · carga: 2 s · memoria máxima 2.87 GB
1,286,487 filas · clase de [201909, 201910]: 100 % NULL


## 4. Clasificación por tipo y meses rotos

La misma clasificación del `02`. Los 23 meses rotos pasan a NA en todo su mes, antes de la historia, **incluidos los 12 de 201910**.

In [4]:
CLAVES = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
VARIABLES = [c for c in base.columns if c not in CLAVES]
BANDERAS = ['active_quarter', 'cliente_vip', 'internet', 'cdescubierto_preacordado',
            'tcallcenter', 'thomebanking', 'tmobile_app', 'ccajas_transacciones']
ESTADOS = ['tcuentas', 'Visa_status', 'Master_status', 'Visa_delinquency', 'Master_delinquency']
FECHAS = [f'{t}_{c}' for t in ('Visa', 'Master') for c in ('Fvencimiento', 'Finiciomora', 'fultimo_cierre', 'fechaalta')]
EDADES = ['cliente_edad', 'cliente_antiguedad']
MONTOS = [v for v in VARIABLES if v not in BANDERAS + ESTADOS + FECHAS + EDADES]
EN_PESOS = [v for v in MONTOS if v.startswith(('m', 'Visa_m', 'Master_m'))]
assert len(VARIABLES) == 152 and (len(MONTOS), len(EN_PESOS)) == (129, 73)
for v in BANDERAS:
    assert set(base[v].dropna().unique()) <= {0, 1}, v

rotos = pd.read_csv(ROTOS_CSV)
assert len(rotos) == ESPERADO_ROTOS and (rotos['foto_mes'] == 201910).sum() == ESPERADO_ROTOS_201910
if TRATAR_ROTOS:
    for v, mes in rotos[['variable', 'foto_mes']].itertuples(index=False):
        base.loc[base['foto_mes'] == mes, v] = np.nan
print(f'Meses rotos: {"a NA" if TRATAR_ROTOS else "sin tratar"} ({len(rotos)} casos)')

# Matrices globales: el valor del mes (columnas contiguas) y la clave cliente-período
FOTO_MES = base['foto_mes'].to_numpy()
PERIODO = (FOTO_MES // 100 * 12 + FOTO_MES % 100).astype(np.int64)
CLAVE = base['numero_de_cliente'].to_numpy(np.int64) * 100_000 + PERIODO
INDICE = pd.Index(CLAVE)
assert INDICE.is_unique
X_MES = np.asfortranarray(base[VARIABLES].to_numpy(np.float32))
CLASE = base['clase_ternaria'].to_numpy()
VARIABLES_CONTROL = ['ctrx_quarter', 'thomebanking', 'Visa_status', 'cpayroll_trx', 'ccomisiones_otras']
CONTROL = base[['numero_de_cliente'] + VARIABLES_CONTROL].assign(p=PERIODO)
del base
gc.collect()
_X_RANKING = None


def x_ranking():
    """Montos en pesos como percentil dentro del mes; el resto igual. Se calcula una vez."""
    global _X_RANKING
    if _X_RANKING is None:
        _X_RANKING = X_MES.copy(order='F')
        idx = [VARIABLES.index(v) for v in EN_PESOS]
        rangos = pd.DataFrame(X_MES[:, idx]).groupby(FOTO_MES).rank(pct=True).to_numpy(np.float32)
        _X_RANKING[:, idx] = rangos
    return _X_RANKING

Meses rotos: a NA (23 casos)


## 5. Tabla de variables de una configuración

El mismo cálculo del `02`, variable por variable para ahorrar memoria:
- lags por mes calendario;
- delta = mes − lag; "cambió" para los estados;
- tendencias con la ventana de m − h a m;
- permanencia para las banderas.

Filas: los meses de entrenamiento de la configuración y después 201908, 201909 y 201910, siempre en el mismo orden.

**Controles de cada tabla:** columnas, cobertura, delta de la antigüedad, lags por otro camino, tendencias a mano y meses rotos.

In [5]:
def especificacion(variante):
    ex = VARIANTES[variante]
    e = [('original', v, 0) for v in VARIABLES]
    e += [(op, v, k) for k in (1, 2) for op in ('lag', 'delta') for v in VARIABLES]
    for k in ex['lags']:
        e += [(op, v, k) for op in ('lag', 'delta') for v in MONTOS + BANDERAS]
        e += [(op, v, k) for op in ('lag', 'cambio') for v in ESTADOS]
    if ex['tendencia']:
        h = ex['tendencia']
        e += [(op, v, h) for op in ('pend', 'prom', 'min', 'max') for v in MONTOS]
        e += [('perm', v, h) for v in BANDERAS]
    return e


def nombre(op, v, k):
    return v if op == 'original' else f'{v}_{op}{k}'


def construir(variante, meses_train, historia):
    ex = VARIANTES[variante]
    h = ex['tendencia']
    h_max = max([2] + ex['lags'] + [h or 0])
    filas = np.flatnonzero(np.isin(FOTO_MES, meses_train + MESES_EVAL))
    mes_fila = FOTO_MES[filas]
    n_train = int(np.isin(mes_fila, meses_train).sum())
    assert np.isin(mes_fila[:n_train], meses_train).all() and np.isin(mes_fila[n_train:], MESES_EVAL).all()
    pos = {k: INDICE.get_indexer(CLAVE[filas] - k) for k in range(h_max + 1)}
    X = x_ranking() if historia == 'ranking' else X_MES

    espec = especificacion(variante)
    ubic = {e: i for i, e in enumerate(espec)}
    F = np.empty((len(filas), len(espec)), dtype=np.float32)
    t = -np.arange((h or 0) + 1, dtype=np.float64)[:, None]
    with np.errstate(invalid='ignore', divide='ignore'):
        for j, v in enumerate(VARIABLES):
            F[:, ubic[('original', v, 0)]] = X_MES[filas, j]
            hk = {}
            for k in range(h_max + 1):
                col = X[pos[k], j]
                col[pos[k] < 0] = np.nan
                hk[k] = col
            for k in range(1, h_max + 1):
                if ('lag', v, k) in ubic:
                    F[:, ubic[('lag', v, k)]] = hk[k]
                if ('delta', v, k) in ubic:
                    F[:, ubic[('delta', v, k)]] = hk[0] - hk[k]
                if ('cambio', v, k) in ubic:
                    a, b = hk[0], hk[k]
                    F[:, ubic[('cambio', v, k)]] = np.where(np.isnan(a) | np.isnan(b), np.nan, (a != b).astype(np.float32))
            if h and (('prom', v, h) in ubic or ('perm', v, h) in ubic):
                S = np.stack([hk[k] for k in range(h + 1)]).astype(np.float64)
                valido = ~np.isnan(S)
                cuantos = valido.sum(axis=0)
                S0 = np.where(valido, S, 0.0)
                if ('perm', v, h) in ubic:
                    F[:, ubic[('perm', v, h)]] = np.where(cuantos > 0, S0.sum(axis=0), np.nan)
                else:
                    prom = S0.sum(axis=0) / cuantos
                    dt = np.where(valido, t - (valido * t).sum(axis=0) / cuantos, 0.0)
                    pend = (dt * (S0 - prom)).sum(axis=0) / (dt ** 2).sum(axis=0)
                    pend[cuantos < 2] = np.nan
                    F[:, ubic[('pend', v, h)]] = pend
                    F[:, ubic[('prom', v, h)]] = prom
                    F[:, ubic[('min', v, h)]] = np.where(cuantos > 0, np.where(valido, S, np.inf).min(axis=0), np.nan)
                    F[:, ubic[('max', v, h)]] = np.where(cuantos > 0, np.where(valido, S, -np.inf).max(axis=0), np.nan)
    return {'F': F, 'columnas': [nombre(*e) for e in espec], 'filas': filas, 'mes_fila': mes_fila,
            'n_train': n_train, 'lags': sorted({k for op, _, k in espec if op == 'lag'}), 'h': h}


cobertura = pd.read_csv(COBERTURA_CSV, index_col='foto_mes')


def controlar(T, variante, historia):
    F, col, mes_fila = T['F'], {c: i for i, c in enumerate(T['columnas'])}, T['mes_fila']
    assert F.shape[1] == ESPERADO_N_VARIABLES[variante]
    for k in [k for k in T['lags'] if k >= 2]:   # lags contra la cobertura
        pct = pd.Series(~np.isnan(F[:, col[f'{VARIABLE_COBERTURA}_lag{k}']])).groupby(mes_fila).mean().mul(100)
        for mes, valor in pct.items():
            assert abs(valor - cobertura.loc[mes, f'h{k}']) <= 0.051, (k, mes, valor)
    d1, l1 = F[:, col['cliente_antiguedad_delta1']], F[:, col['cliente_antiguedad_lag1']]
    assert (d1[~np.isnan(l1)] == 1).all()

    rng = np.random.default_rng(SEMILLA_PRIMIGENIA)   # lags por otro camino (variables fuera de EN_PESOS)
    muestra = rng.choice(len(T['filas']), size=2_000, replace=False)
    claves = CONTROL.iloc[T['filas'][muestra]][['numero_de_cliente', 'p']]
    for k in T['lags']:
        buscado = claves.assign(p=claves['p'] - k).merge(CONTROL, on=['numero_de_cliente', 'p'], how='left')
        for v in VARIABLES_CONTROL:
            np.testing.assert_array_equal(F[muestra, col[f'{v}_lag{k}']], buscado[v].to_numpy(np.float32))

    h = T['h']
    if h:   # tendencias a mano
        v = 'ctrx_quarter'
        valores = pd.Series(CONTROL[v].to_numpy(np.float64), index=INDICE)
        ventana = np.stack([valores.reindex(CLAVE[T['filas'][muestra]] - k).to_numpy() for k in range(h + 1)])
        for j in rng.choice(np.flatnonzero(~np.isnan(ventana).any(axis=0)), size=3, replace=False):
            i, y = muestra[j], ventana[:, j]
            np.testing.assert_allclose(
                [F[i, col[f'{v}_pend{h}']], F[i, col[f'{v}_prom{h}']], F[i, col[f'{v}_min{h}']], F[i, col[f'{v}_max{h}']]],
                [np.polyfit(-np.arange(h + 1), y, 1)[0], y.mean(), y.min(), y.max()], rtol=1e-4, atol=1e-4)

    if TRATAR_ROTOS:   # meses rotos: NA en su mes y en los lags que lo miran
        periodo_fila = mes_fila // 100 * 12 + mes_fila % 100
        for v, mes in rotos[['variable', 'foto_mes']].itertuples(index=False):
            for k in [0] + T['lags']:
                columna = v if k == 0 else f'{v}_lag{k}'
                en_destino = periodo_fila == p0(mes) + k
                if en_destino.any() and columna in col:
                    assert np.isnan(F[en_destino, col[columna]]).all(), (v, mes, k)


def huella_control(T):
    """Huella de las 760 columnas de A (filas incluidas), para comparar entre configuraciones."""
    hsh = hashlib.blake2b(digest_size=16)
    hsh.update(T['filas'].tobytes())
    for j in range(ESPERADO_N_VARIABLES['A']):
        hsh.update(T['F'][:, j].tobytes())
    return hsh.hexdigest()

## 6. Optuna sobre `A` (50 pruebas)

- La misma búsqueda del `02`: los 5 hiperparámetros de la consigna, AUC en 201908 contra `BAJA+2` y *early stopping*.
- Con `PRIMERA_PRUEBA_02`, la prueba 0 es la mejor del `02`.
- **Si ya existe `resultados/03_hiperparametros.json`, no se corre**: se usan esos. Así funciona la reproducción cruzada en otra máquina.

Los hiperparámetros elegidos se usan en **las 10 configuraciones**.

In [6]:
FIJOS = {'objective': 'binary', 'metric': 'auc', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1,
         'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0, 'feature_pre_filter': False,
         'num_threads': N_HILOS, 'deterministic': True, 'force_col_wise': True, 'verbose': -1}


def datos(T):
    n = T['n_train']
    y_train = np.isin(CLASE[T['filas'][:n]], POSITIVOS).astype(np.int8)
    return lgb.Dataset(T['F'][:n], label=y_train, feature_name=T['columnas'],
                       params={'max_bin': 31, 'feature_pre_filter': False}, free_raw_data=False)


if HIPER_JSON.exists():
    HIPER = json.loads(HIPER_JSON.read_text())
    print(f'Se usan los hiperparámetros guardados ({HIPER_JSON}): prueba {HIPER["prueba"]}, '
          f'AUC {HIPER["auc_201908"]:.5f}, {HIPER["arboles"]} árboles, calculados en {HIPER["maquina"]}')
else:
    t0 = time.time()
    variante, meses, historia = CONFIG['A']
    T = construir(variante, meses, historia)
    controlar(T, variante, historia)
    dtrain = datos(T)
    es_valid = T['mes_fila'] == MES_VALID
    dvalid = lgb.Dataset(T['F'][es_valid], label=np.isin(CLASE[T['filas'][es_valid]], POSITIVOS_AUC).astype(np.int8),
                         reference=dtrain, free_raw_data=False)

    def objetivo(trial):
        p = {**FIJOS,
             'num_leaves': trial.suggest_int('num_leaves', 8, 512, log=True),
             'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 2000, log=True),
             'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
             'feature_fraction': trial.suggest_float('feature_fraction', 0.2, 1.0),
             'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
             'seed': SEMILLAS[0]}
        t1 = time.time()
        modelo = lgb.train(p, dtrain, num_boost_round=ARBOLES_MAX, valid_sets=[dvalid], valid_names=['valid'],
                           callbacks=[lgb.early_stopping(PACIENCIA, verbose=False)])
        auc = modelo.best_score['valid']['auc']
        trial.set_user_attr('arboles', modelo.best_iteration)
        trial.set_user_attr('segundos', round(time.time() - t1, 1))
        print(f'prueba {trial.number}: AUC {auc:.5f} · {modelo.best_iteration} árboles · hojas {p["num_leaves"]} · '
              f'mdl {p["min_data_in_leaf"]} · lr {p["learning_rate"]:.3f} · ff {p["feature_fraction"]:.2f} · '
              f'bf {p["bagging_fraction"]:.2f} · {time.time() - t1:.0f} s')
        return auc

    estudio = optuna.create_study(study_name='03_A', storage=STORAGE, direction='maximize', load_if_exists=True,
                                  sampler=optuna.samplers.TPESampler(seed=SEMILLA_PRIMIGENIA))
    if PRIMERA_PRUEBA_02 and len(estudio.trials) == 0:
        estudio.enqueue_trial(MEJOR_02)
    hechas = sum(t.state == optuna.trial.TrialState.COMPLETE for t in estudio.trials)
    estudio.optimize(objetivo, n_trials=max(0, N_PRUEBAS_OPTUNA - hechas))
    completas = [t for t in estudio.trials if t.state == optuna.trial.TrialState.COMPLETE]
    HIPER = {'prueba': estudio.best_trial.number, 'params': estudio.best_trial.params,
             'arboles': estudio.best_trial.user_attrs['arboles'], 'auc_201908': estudio.best_value,
             'pruebas': len(completas), 'segundos_por_prueba': float(np.mean([t.user_attrs['segundos'] for t in completas])),
             'maquina': platform.node(), 'fecha_hora': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
    HIPER_JSON.write_text(json.dumps(HIPER, indent=2))
    del T, dtrain, dvalid
    gc.collect()
    registrar('A', 'optuna', t0)

MEJORES = {**FIJOS, **HIPER['params']}
ARBOLES = HIPER['arboles']
print({k: round(v, 4) if isinstance(v, float) else v for k, v in HIPER['params'].items()}, f'· {ARBOLES} árboles')

prueba 0: AUC 0.90928 · 370 árboles · hojas 77 · mdl 1147 · lr 0.016 · ff 0.56 · bf 0.98 · 41 s
prueba 1: AUC 0.90607 · 724 árboles · hojas 71 · mdl 11 · lr 0.011 · ff 0.38 · bf 0.68 · 47 s
prueba 2: AUC 0.90787 · 328 árboles · hojas 87 · mdl 468 · lr 0.013 · ff 0.76 · bf 0.83 · 36 s
prueba 3: AUC 0.90705 · 415 árboles · hojas 275 · mdl 1230 · lr 0.013 · ff 0.44 · bf 0.75 · 46 s
prueba 4: AUC 0.90343 · 43 árboles · hojas 66 · mdl 676 · lr 0.079 · ff 0.37 · bf 0.74 · 9 s
prueba 5: AUC 0.90563 · 700 árboles · hojas 19 · mdl 20 · lr 0.013 · ff 0.37 · bf 0.47 · 29 s
prueba 6: AUC 0.89726 · 55 árboles · hojas 243 · mdl 201 · lr 0.056 · ff 0.48 · bf 0.45 · 17 s
prueba 7: AUC 0.89907 · 45 árboles · hojas 342 · mdl 573 · lr 0.099 · ff 0.77 · bf 0.74 · 25 s
prueba 8: AUC 0.90154 · 149 árboles · hojas 340 · mdl 86 · lr 0.046 · ff 0.23 · bf 0.92 · 30 s
prueba 9: AUC 0.90210 · 126 árboles · hojas 41 · mdl 15 · lr 0.035 · ff 0.65 · bf 0.56 · 10 s
prueba 10: AUC 0.90544 · 391 árboles · hojas 9 · mdl

## 7. Corridas

Para cada configuración: se arma su tabla, se controla y se entrenan las semillas que faltan. Cada semilla:
- predice 201908, 201909 y 201910, y guarda las probabilidades en `exp/03/pred/<config>/<semilla>.npy`;
- calcula **solo en 201908** la AUC y la curva de ganancia, que se usa para elegir N.

**Control:** las 760 columnas de A tienen que ser idénticas (huella) en las 7 configuraciones nominales de 201905–06. A con la primera semilla tiene que reproducir la AUC de Optuna.

In [7]:
def curva_ganancia(prob, es_b2, largo=LARGO_CURVA):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_b2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:largo].astype(np.float64)


huellas = json.loads(HASHES.read_text()) if HASHES.exists() else {}

for config in CONFIGURACIONES:
    variante, meses, historia = CONFIG[config]
    (PRED / config).mkdir(exist_ok=True)
    (CURVAS / config).mkdir(exist_ok=True)
    pendientes = [s for s in SEMILLAS_A_USAR if not (PRED / config / f'{s}.npy').exists()]
    if not pendientes:
        print(f'{config}: las {len(SEMILLAS_A_USAR)} semillas ya estaban')
        continue

    t0 = time.time()
    T = construir(variante, meses, historia)
    controlar(T, variante, historia)
    eval_ = slice(T['n_train'], None)
    claves = {'numero_de_cliente': (CLAVE[T['filas'][eval_]] // 100_000), 'foto_mes': T['mes_fila'][eval_]}
    if CLAVES_EVAL.exists():
        guardadas = np.load(CLAVES_EVAL)
        assert all((guardadas[c] == claves[c]).all() for c in claves), 'Las filas de evaluación cambiaron'
    else:
        np.savez(CLAVES_EVAL, **claves)
    if config in GRUPO_CONTROL:
        huellas[config] = huella_control(T)
        HASHES.write_text(json.dumps(huellas, indent=2))
        assert len(set(huellas.values())) == 1, f'Las columnas de A difieren entre configuraciones: {huellas}'
    dtrain = datos(T)
    es_valid = T['mes_fila'][eval_] == MES_VALID
    clase_valid = CLASE[T['filas'][eval_][es_valid]]
    y_auc, es_baja2 = np.isin(clase_valid, POSITIVOS_AUC), clase_valid == 'BAJA+2'
    registrar(config, 'tabla y controles', t0)
    print(f'{config}: {T["n_train"]:,} filas de entrenamiento · {T["F"].shape[1]:,} variables')

    for semilla in pendientes:
        t0 = time.time()
        modelo = lgb.train({**MEJORES, 'seed': semilla}, dtrain, num_boost_round=ARBOLES)
        prob = modelo.predict(T['F'][eval_], num_threads=N_HILOS).astype(np.float32)
        auc = roc_auc_score(y_auc, prob[es_valid])
        assert 0.8 < auc < 0.98, f'{config} · {semilla}: AUC {auc:.4f} fuera de lo razonable'
        if config == 'A' and semilla == SEMILLAS[0]:
            if HIPER['maquina'] == platform.node():
                assert abs(auc - HIPER['auc_201908']) < 1e-6, (auc, HIPER['auc_201908'])
            print(f'Reproducción: AUC {auc:.6f} · la de Optuna ({HIPER["maquina"]}): {HIPER["auc_201908"]:.6f}')
        np.save(CURVAS / config / f'{semilla}.npy', curva_ganancia(prob[es_valid], es_baja2))
        fila = pd.DataFrame([{'config': config, 'variante': variante, 'historia': historia,
                              'meses_train': f'{meses[0]}-{meses[-1]}', 'semilla': semilla,
                              'n_columnas': T['F'].shape[1], 'arboles': ARBOLES, 'auc_201908': round(auc, 6),
                              'segundos': round(time.time() - t0, 1), 'maquina': platform.node()}])
        fila.to_csv(VALIDACION_CSV, mode='a', header=not VALIDACION_CSV.exists(), index=False)
        np.save(PRED / config / f'{semilla}.npy', prob)   # al final: si existe, la semilla está completa
        print(f'{config} · semilla {semilla}: AUC 201908 {auc:.5f} · {time.time() - t0:.0f} s')
    del T, dtrain, modelo
    gc.collect()

A · tabla y controles: 8 s · memoria máxima 3.11 GB
A: 255,133 filas de entrenamiento · 760 variables
Reproducción: AUC 0.909510 · la de Optuna (MBA-M4): 0.909510
A · semilla 294103: AUC 201908 0.90951 · 23 s
A · semilla 342929: AUC 201908 0.90602 · 21 s
A · semilla 161093: AUC 201908 0.90703 · 21 s
A · semilla 677561: AUC 201908 0.90839 · 21 s
A · semilla 431077: AUC 201908 0.90749 · 21 s
A · semilla 330679: AUC 201908 0.90614 · 21 s
A · semilla 511579: AUC 201908 0.90880 · 21 s
A · semilla 312773: AUC 201908 0.90767 · 21 s
A · semilla 118147: AUC 201908 0.90651 · 21 s
A · semilla 567529: AUC 201908 0.90778 · 21 s
L3 · tabla y controles: 9 s · memoria máxima 3.75 GB
L3: 255,133 filas de entrenamiento · 1,044 variables
L3 · semilla 294103: AUC 201908 0.90626 · 31 s
L3 · semilla 342929: AUC 201908 0.90690 · 27 s
L3 · semilla 161093: AUC 201908 0.90731 · 28 s
L3 · semilla 677561: AUC 201908 0.90682 · 27 s
L3 · semilla 431077: AUC 201908 0.90589 · 27 s
L3 · semilla 330679: AUC 201908 0.90

## 8. N por configuración (solo con 201908)

Para cada configuración:
1. se promedia la curva de ganancia de 201908 de sus semillas;
2. se suaviza con una media móvil centrada de 501;
3. **N** es el máximo de esa curva.

Ese N es el que el `04` aplica a 201909 y 201910. Se agrega también la ganancia de cada semilla en ese N (valor real de la curva, sin suavizar).

In [8]:
validacion = pd.read_csv(VALIDACION_CSV).drop_duplicates(['config', 'semilla'], keep='last')
filas_n = []
for config in CONFIGURACIONES:
    curvas = np.stack([np.load(CURVAS / config / f'{s}.npy') for s in SEMILLAS_A_USAR])
    suave = pd.Series(curvas.mean(axis=0)).rolling(VENTANA_SUAVIZADO, center=True,
                                                   min_periods=VENTANA_SUAVIZADO).mean().to_numpy()
    n = int(np.nanargmax(suave)) + 1
    por_semilla = curvas[:, n - 1] / 1e6
    for s, g in zip(SEMILLAS_A_USAR, por_semilla):
        validacion.loc[(validacion['config'] == config) & (validacion['semilla'] == s), 'ganancia_201908_en_N_M'] = round(g, 3)
    filas_n.append({'config': config, 'N': n, 'semillas': len(SEMILLAS_A_USAR),
                    'ganancia_201908_media_M': round(por_semilla.mean(), 3),
                    'ganancia_201908_desvio_M': round(por_semilla.std(ddof=1), 3) if len(por_semilla) > 1 else None})
validacion.to_csv(VALIDACION_CSV, index=False)
tabla_n = pd.DataFrame(filas_n)
tabla_n.to_csv(N_CSV, index=False)

completo = validacion[validacion['config'].isin(CONFIGURACIONES) & validacion['semilla'].isin(SEMILLAS_A_USAR)]
assert len(completo) == len(CONFIGURACIONES) * len(SEMILLAS_A_USAR)
assert all((PRED / c / f'{s}.npy').exists() for c in CONFIGURACIONES for s in SEMILLAS_A_USAR)
resumen = completo.groupby('config').agg(variables=('n_columnas', 'first'), auc_201908=('auc_201908', 'mean'))
print('Solo validación (201908). 201909 y 201910 se evalúan en el 04.')
print(tabla_n.set_index('config').join(resumen).loc[CONFIGURACIONES].round(4).to_string())

Solo validación (201908). 201909 y 201910 se evalúan en el 04.
                N  semillas  ganancia_201908_media_M  ganancia_201908_desvio_M  variables  auc_201908
config                                                                                               
A            3855        10                  197.808                     2.061        760      0.9075
L3           4221        10                  198.412                     3.719       1044      0.9070
L4           3672        10                  195.360                     3.818       1328      0.9081
T3           3975        10                  200.888                     4.213       1284      0.9074
T4           4260        10                  199.650                     3.154       1284      0.9089
LT3          4046        10                  200.145                     2.227       1568      0.9075
LT4          4209        10                  198.082                     4.278       1852      0.9082
A-ranking    3934  

## 9. Tiempos y cierre

- **Verificación:** si el notebook llegó hasta acá, se cumplen los controles de la ficha (filas, ceguera, columnas, huella de A, cobertura, meses rotos, reproducción y completitud).
- **Lo que sigue:** el verificador, en una sesión nueva. Después, el `04`: la regla pre-registrada, una sola vez.

In [9]:
seg = pd.read_csv(VALIDACION_CSV).groupby('config')['segundos'].mean()
for config in CONFIGURACIONES:
    TIEMPOS.append({'config': config, 'paso': 'un modelo (media)', 'segundos': round(seg[config], 1),
                    'memoria_max_gb': memoria_gb()})
tiempos = pd.DataFrame(TIEMPOS).assign(fecha_hora=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
                                       maquina=platform.node(), hilos=N_HILOS)
tiempos.to_csv(TIEMPOS_CSV, mode='a', header=not TIEMPOS_CSV.exists(), index=False)
print(tiempos[['config', 'paso', 'segundos', 'memoria_max_gb']].to_string(index=False))

     config              paso  segundos  memoria_max_gb
          -             carga       1.7            2.87
          A            optuna    1649.9            3.11
          A tabla y controles       7.8            3.11
         L3 tabla y controles       9.5            3.75
         L4 tabla y controles      12.1            4.61
         T3 tabla y controles      12.1            4.68
         T4 tabla y controles      13.3            4.68
        LT3 tabla y controles      14.5            5.08
        LT4 tabla y controles    2155.9            5.11
  A-ranking tabla y controles      12.5            5.11
LT4-ranking tabla y controles    1413.7            5.11
 A-consigna tabla y controles       4.7            5.70
          A un modelo (media)      21.5            5.70
         L3 un modelo (media)      27.5            5.70
         L4 un modelo (media)      33.6            5.70
         T3 un modelo (media)      32.7            5.70
         T4 un modelo (media)      33.3         